In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ z_1=xW_1+b_1 $$
$$ p_1 = \frac{e^{z_1}}{e^{z_1} + 1} $$
$$ z_2=p_1W_2+b_2 $$
$$ p_2 = \frac{e^{z_2}}{e^{z_2} + 1} $$
$$ L = -y \log(p_2) - (1-y) \log(1-p_2) $$

$$ \diamonds \diamonds \diamonds $$

$$ \frac{\partial L}{\partial z_2} = \frac{\partial L}{\partial p_2} \frac{\partial p_2}{\partial z_2} = \Bigg( \frac{p_2-y}{p_2(1-p_2)} \Bigg) \Bigg( p_2(1-p_2) \Bigg) = p_2-y $$
$$ \frac{\partial L}{\partial W_2} = \frac{\partial L}{\partial z_2} \frac{\partial z_2}{\partial W_2} = (p_2-y) p_1 $$
$$ \frac{\partial L}{\partial b_2} = \frac{\partial L}{\partial z_2} \frac{\partial z_2}{\partial b_2} = (p_2-y) $$
$$ \frac{\partial L}{\partial p_1} = \frac{\partial L}{\partial z_2} \frac{\partial z_2}{\partial p_1} = (p_2-y) W_2^T $$
$$ \frac{\partial p_1}{\partial z_1} = p_1(1-p_1) $$
$$ \frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial p_1} \frac{\partial p_1}{\partial z_1} = \Big( (p_2-y) W_2^T \Big) \Big( p_1(1-p_1) \Big) $$
$$ \frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial z_1} \frac{\partial z_1}{\partial W_1} = x^T \Big( (p_2-y) W_2^T p_1(1-p_1) \Big) $$
$$ \frac{\partial L}{\partial b_1} = \frac{\partial L}{\partial z_1} \frac{\partial z_1}{\partial b_1} = (p_2-y) W_2^T p_1(1-p_1)$$
$$ \frac{\partial L}{\partial x} = \frac{\partial L}{\partial z_1} \frac{\partial z_1}{\partial x} = \Big( (p_2-y) W_2^T p_1(1-p_1) \Big) W_1^T $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd

import import_ipynb
import sigmoid # type: ignore
import linear # type: ignore
import binary_cross_entropy # type: ignore
from common import assert_eq, average_run, test_perceptron_boolean # type: ignore


class Per_Lin_Sig_Lin_Sig_BCE_Gradient_Function(autograd.Function):
    @staticmethod
    def forward(ctx, x, W1, b1, W2, b2, y):
        z1 = linear.linear(x, W1, b1)      # (S, FH)
        p1 = sigmoid.sigmoid(z1)           # (S, FH)
        z2 = linear.linear(p1, W2, b2)     # (S, FO)
        p2 = sigmoid.sigmoid(z2)           # (S, FO)

        L = binary_cross_entropy.binary_cross_entropy(p2, y)

        ctx.save_for_backward(x, W1, b1, W2, b2, y, z1, p1, z2, p2)

        return L

    @staticmethod
    def backward(ctx, dF_dL):
        (x, W1, b1, W2, b2, y, z1, p1, z2, p2) = ctx.saved_tensors

        S = x.shape[0]   # Samples
        FI = x.shape[1]  # Features In
        FH = W1.shape[1] # Features Hidden
        FO = W2.shape[1] # Features Out

        assert_eq(x.shape,  (S, FI))
        assert_eq(W1.shape, (FI, FH))
        assert_eq(b1.shape, (FH,))
        assert_eq(W2.shape, (FH, FO))
        assert_eq(b2.shape, (FO,))
        assert_eq(z1.shape, (S, FH))
        assert_eq(p1.shape, (S, FH))
        assert_eq(z2.shape, (S, FO))
        assert_eq(p2.shape, (S, FO))
        assert_eq(y.shape,  (S, FO))

        #
        # 2nd layer
        #

        dL_dz2 = (p2 - y) / S
        assert_eq(dL_dz2.shape, (S, FO))

        # (FH, FO) = (S, FH).T @ (S, FO)
        dL_dW2 = p1.t() @ dL_dz2                
        assert_eq(dL_dW2.shape, (FH, FO))

        dL_db2 = dL_dz2.sum(dim=0)              
        assert_eq(dL_db2.shape, (FO,))

        # (S, FH) = (S, FO) @ (FH, FO).T
        dL_dp1 = dL_dz2 @ W2.t()                
        assert_eq(dL_dp1.shape, (S, FH))

        #
        # 1st sigmoid
        #
        
        dp1_dz1 = p1 * (1 - p1)                  
        assert_eq(dp1_dz1.shape, (S, FH))

        dL_dz1 = dL_dp1 * dp1_dz1               
        assert_eq(dL_dz1.shape, (S, FH))

        #
        # 1st layer
        #

        # (FI, FH) = (S, FI).T @ (S, FH)
        dL_dW1 = x.t() @ dL_dz1                  
        assert_eq(dL_dW1.shape, (FI, FH))

        dL_db1 = dL_dz1.sum(dim=0)             
        assert_eq(dL_db1.shape, (FH,))

        # (S, FI) = (S, FH) @ (FI, FH).T
        dL_dx = dL_dz1 @ W1.t()                  
        assert_eq(dL_dx.shape, (S, FI))

        return (dF_dL * dL_dx, 
                dF_dL * dL_dW1, 
                dF_dL * dL_db1, 
                dF_dL * dL_dW2, 
                dF_dL * dL_db2, 
                None)
    

class Per_Lin_Sig_Lin_Sig_BCE_Gradient(nn.Module):
    def __init__(self, in_features: int, hidden_features: int, out_features: int):
        super().__init__()

        self.weight1 = nn.Parameter(tr.randn(hidden_features, in_features, dtype=tr.float32))
        self.bias1 = nn.Parameter(tr.randn(hidden_features, dtype=tr.float32))
        self.weight2 = nn.Parameter(tr.randn(out_features, hidden_features, dtype=tr.float32))
        self.bias2 = nn.Parameter(tr.randn(out_features, dtype=tr.float32))

    def forward(self, x, y):
        return Per_Lin_Sig_Lin_Sig_BCE_Gradient_Function.apply(
            x, 
            self.weight1.T, 
            self.bias1, 
            self.weight2.T, 
            self.bias2, 
            y
        )
    
    # Extension
    # To facilitate testing and experimentation various perceptrons.
    def learn(self, x, y):
        return self(x, y)

    # Extension
    # To facilitate testing and experimentation various perceptrons.
    def predict(self, x):
        with tr.no_grad():
            z1 = linear.LinearFunction.apply(x, self.weight1.T, self.bias1)
            p1 = sigmoid.SigmoidFunction.apply(z1)
            z2 = linear.LinearFunction.apply(p1, self.weight2.T, self.bias2)
            p2 = sigmoid.SigmoidFunction.apply(z2)
            return p2 >= 0.5


def test_Per_Lin_Sig_Lin_Sig_BCE_Gradient(bool_fn, epochs, lr=0.1):
    return test_perceptron_boolean(Per_Lin_Sig_Lin_Sig_BCE_Gradient(2, 4, 1), bool_fn, epochs=epochs, lr=lr)


if __name__ == "__main__":
    N=3

    logical_and = tr.logical_and
    print(f" AND, {5*N} epochs: {average_run(1*N, lambda: test_Per_Lin_Sig_Lin_Sig_BCE_Gradient(logical_and, epochs=5*N)):.2f}")

    logical_or = tr.logical_or
    print(f"  OR, {5*N} epochs: {average_run(1*N, lambda: test_Per_Lin_Sig_Lin_Sig_BCE_Gradient(logical_or, epochs=5*N)):.2f}")

    logical_nand = lambda x1, x2: tr.logical_not(tr.logical_and(x1, x2))
    print(f"NAND, {5*N} epochs: {average_run(1*N, lambda: test_Per_Lin_Sig_Lin_Sig_BCE_Gradient(logical_nand, epochs=5*N)):.2f}")

    logical_xor = lambda x1, x2: tr.logical_xor(x1, x2)
    print(f" XOR, {5*N} epochs: {average_run(1*N, lambda: test_Per_Lin_Sig_Lin_Sig_BCE_Gradient(logical_xor, epochs=5*N)):.2f}")


 AND, 15 epochs: 0.63
  OR, 15 epochs: 0.77
NAND, 15 epochs: 0.65
 XOR, 15 epochs: 0.64
